## HTML files analysis

### Load

In [17]:
import pandas as pd
import numpy as np
import ast

CSV_PATH = "../data/evaluation/metadata_html_samples_cleaned.csv"

df = pd.read_csv(CSV_PATH, dtype={"internal_id": "string"})

# Converti query_params da string a dict
def safe_eval(x):
    try:
        return ast.literal_eval(x)
    except:
        return {}

df["query_params"] = df["query_params"].apply(safe_eval)

# === CONVERSIONE SIZE IN KB ===
df["size_kb"] = df["size"] / 1024
df["size_kb"] = df["size_kb"].round(2)

### STATISTICHE GENERALI

In [18]:
print(" - Numero totale documenti:", len(df))
print(" -  Numero domini unici:", df["domain"].nunique())
print(" -  Numero internal_id unici:", df["internal_id"].nunique())
print(" -  Numero base_path unici:", df["base_path"].nunique())

 - Numero totale documenti: 10772
 -  Numero domini unici: 4
 -  Numero internal_id unici: 165
 -  Numero base_path unici: 262


### DOCUMENTI PER DOMINIO

In [19]:
print("\n==============================")
print(" DOCUMENTI PER DOMINIO")
print("==============================\n")

print(df["domain"].value_counts().head(10))


 DOCUMENTI PER DOMINIO

domain
docenti.unisa.it          9581
www.diem.unisa.it          939
corsi.unisa.it             251
www.diem.unisa.it.html       1
Name: count, dtype: int64


# DISTRIBUZIONE DEPTH

In [20]:
print("\n==============================")
print(" DISTRIBUZIONE DEPTH")
print("==============================\n")

print(df["depth"].value_counts().sort_index())


 DISTRIBUZIONE DEPTH

depth
0     174
1     505
2    2291
3    3812
4    2050
5    1940
Name: count, dtype: int64


### ID (Matricole)

In [21]:
print("\n==============================")
print(" TOP INTERNAL_ID")
print("==============================\n")

print(df["internal_id"].value_counts().head(10))


 TOP INTERNAL_ID

internal_id
001295    464
005768    373
005501    344
003741    336
023586    271
005630    261
004491    253
001366    242
023604    240
004687    233
Name: count, dtype: int64[pyarrow]


### ANALISI DIMENSIONI FILE

In [28]:
print("\n==============================")
print(" ANALISI DIMENSIONI FILE (KB)")
print("==============================\n")

percentiles = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]

size_stats = df["size_kb"].describe(percentiles=percentiles).round(2)
print(size_stats)

print("\n- Media (KB):", round(df["size_kb"].mean(), 2))
print("- Mediana (KB):", round(df["size_kb"].median(), 2))
print("- Std Dev (KB):", round(df["size_kb"].std(), 2))



 ANALISI DIMENSIONI FILE (KB)

count    10772.00
mean         6.79
std         12.09
min          0.18
1%           0.18
5%           0.68
10%          0.84
25%          1.65
50%          3.89
75%         10.12
90%         13.68
95%         16.64
99%         35.62
max        675.58
Name: size_kb, dtype: float64

- Media (KB): 6.79
- Mediana (KB): 3.89
- Std Dev (KB): 12.09


In [23]:
print("\n==============================")
print(" DISTRIBUZIONE DIMENSIONI (BINNING)")
print("==============================\n")

bins = [0, 1, 2, 3, 4, 5, 10, 20, 50, 100, np.inf]
labels = [
    "0-1 KB", "1-2 KB", "2-3 KB", "3-4 KB", "4-5 KB",
    "5-10 KB", "10-20 KB", "20-50 KB", "50-100 KB", ">100 KB"
]

df["size_bin"] = pd.cut(df["size_kb"], bins=bins, labels=labels)

bin_counts = df["size_bin"].value_counts().sort_index()
bin_percent = df["size_bin"].value_counts(normalize=True).sort_index() * 100

print("Conteggi:")
print(bin_counts)

print("\nPercentuali:")
print(bin_percent.round(2).astype(str) + " %")


 DISTRIBUZIONE DIMENSIONI (BINNING)

Conteggi:
size_bin
0-1 KB       1216
1-2 KB       2246
2-3 KB       1166
3-4 KB        811
4-5 KB        452
5-10 KB      2128
10-20 KB     2407
20-50 KB      305
50-100 KB      33
>100 KB         8
Name: count, dtype: int64

Percentuali:
size_bin
0-1 KB       11.29 %
1-2 KB       20.85 %
2-3 KB       10.82 %
3-4 KB        7.53 %
4-5 KB         4.2 %
5-10 KB      19.75 %
10-20 KB     22.34 %
20-50 KB      2.83 %
50-100 KB     0.31 %
>100 KB       0.07 %
Name: proportion, dtype: str


### OUTLIER 

In [24]:
print("\n==============================")
print(" FILE PIÙ GRANDI")
print("==============================\n")

largest = df.sort_values(by="size_kb", ascending=False).head(10)
print(largest[["doc_id", "domain", "size_kb"]])


 FILE PIÙ GRANDI

       doc_id             domain  size_kb
10771    2858  www.diem.unisa.it   675.58
10770   13096  www.diem.unisa.it   536.76
10769   12792  www.diem.unisa.it   409.32
10768   31500  www.diem.unisa.it   294.13
10767   12635  www.diem.unisa.it   141.09
10766   31356  www.diem.unisa.it   117.51
10765    4903   docenti.unisa.it   112.61
10764   12646  www.diem.unisa.it   103.37
10763   19363   docenti.unisa.it    95.92
10762   31374  www.diem.unisa.it    94.40


### QUERY PARAMS

In [25]:
print("\n==============================")
print(" ANALISI QUERY PARAMS")
print("==============================\n")

param_counts = {}

for params in df["query_params"]:
    for key in params:
        param_counts[key] = param_counts.get(key, 0) + 1

sorted_params = sorted(param_counts.items(), key=lambda x: x[1], reverse=True)

for k, v in sorted_params[:10]:
    print(f"{k}: {v}")


 ANALISI QUERY PARAMS

anno: 4508
id: 3474
cId: 2500
pId: 2500
ruolo: 2187
progetto: 2023
stato: 1924
tip: 1737
tag: 206
incubatore: 110


### CROSS ANALYSIS

In [26]:
print("\n==============================")
print(" CROSS ANALYSIS (domain vs depth)")
print("==============================\n")

pivot = pd.pivot_table(
    df,
    index="domain",
    columns="depth",
    values="doc_id",
    aggfunc="count",
    fill_value=0
)
print(pivot.head(10))


 CROSS ANALYSIS (domain vs depth)

depth                     0    1     2     3     4     5
domain                                                  
corsi.unisa.it            7  115    90    25     5     9
docenti.unisa.it        166  378  2133  3015  1958  1931
www.diem.unisa.it         0   12    68   772    87     0
www.diem.unisa.it.html    1    0     0     0     0     0


In [27]:
print("\n==============================")
print(" INSIGHT ")
print("==============================\n")

print("Distribuzione depth (normalizzata):")
print(df["depth"].value_counts(normalize=True).sort_index())

print("\nTop domini (%):")
print(df["domain"].value_counts(normalize=True).head(5))


 INSIGHT 

Distribuzione depth (normalizzata):
depth
0    0.016153
1    0.046881
2    0.212681
3    0.353880
4    0.190308
5    0.180097
Name: proportion, dtype: float64

Top domini (%):
domain
docenti.unisa.it          0.889436
www.diem.unisa.it         0.087170
corsi.unisa.it            0.023301
www.diem.unisa.it.html    0.000093
Name: proportion, dtype: float64
